<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-01-logica/09%20-%20Motores%20de%20Inferencia%20Forward%20e%20Backward%20Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining
## Célula de Manufatura Flexível (FMS)

Neste notebook implementamos algoritmos orientados a objetos de **Forward Chaining** (orientado a dados) e **Backward Chaining** (orientado a hipóteses) com rastreamento de trilha de auditoria (*Audit Trail*) para diagnóstico automático e segurança de atuadores na manufatura.

In [ ]:
from dataclasses import dataclass
from typing import Set, Tuple, List, Dict, Optional, Any

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1

class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []

    def adicionar_regra(self, id_r: str, antecedentes: List[str], consequente: str, desc: str, prioridade: int = 1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))

class MotorInferencia:
    def __init__(self, base_conhecimento: BaseConhecimento):
        self.bc = base_conhecimento

    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        """Executa encadeamento para frente até alcançar o ponto fixo."""
        fatos_conhecidos = set(fatos_iniciais)
        historico_disparos = []
        novos_fatos = True
        passo = 1

        while novos_fatos:
            novos_fatos = False
            regras_candidatas = sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True)
            for regra in regras_candidatas:
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico_disparos.append({
                        "Passo": passo,
                        "Regra": regra.id_regra,
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico": regra.descricao_diagnostico
                    })
                    passo += 1
                    novos_fatos = True
                    break
        return fatos_conhecidos, historico_disparos

    def backward_chaining(self, meta: str, fatos_iniciais: Set[str], trilha: Optional[List[str]] = None) -> Tuple[bool, List[str]]:
        """Executa encadeamento para trás para auditar/provar uma hipótese causal."""
        if trilha is None:
            trilha = []

        if meta in fatos_iniciais:
            trilha.append(f"Fato base comprovado na telemetria: '{meta}'")
            return True, trilha

        regras_para_meta = [r for r in self.bc.regras if r.consequente == meta]
        if not regras_para_meta:
            return False, trilha

        for regra in sorted(regras_para_meta, key=lambda r: r.prioridade, reverse=True):
            trilha.append(f"Avaliando regra {regra.id_regra} ({regra.descricao_diagnostico}) para provar '{meta}'")
            todos_antecedentes_provados = True

            for ant in sorted(regra.antecedentes):
                provado, _ = self.backward_chaining(ant, fatos_iniciais, trilha)
                if not provado:
                    todos_antecedentes_provados = False
                    break

            if todos_antecedentes_provados:
                trilha.append(f"✓ Meta '{meta}' PROVADA com sucesso via regra {regra.id_regra}")
                return True, trilha

        return False, trilha

bc = BaseConhecimento()
bc.adicionar_regra("R-01", ["s_topo", "not_s_base"], "erro_geometria", "Inconsistência Geométrica no Sensor ZS-202", 8)
bc.adicionar_regra("R-02", ["erro_geometria", "peca_em_transito"], "alarme_geral_a1", "Disparo do Sinaleiro de Falha HS-302", 9)
bc.adicionar_regra("R-03", ["alarme_geral_a1", "modo_auto"], "trip_esteira_m101", "Desarme de Segurança da Esteira Principal M-101", 10)
bc.adicionar_regra("R-04", ["silo_vazio", "peca_saida"], "conflito_silo", "Conflito Chave de Nível LS-101 e Presença ZS-101", 7)
bc.adicionar_regra("R-05", ["conflito_silo"], "bloqueio_alimentacao_xv101", "Bloqueio do Pistão de Infeed XV-101", 8)

motor = MotorInferencia(bc)

fatos_entrada = {"s_topo", "not_s_base", "peca_em_transito", "modo_auto"}
fatos_finais, trilha_fc = motor.forward_chaining(fatos_entrada)

print("=== TRILHA DE DIAGNÓSTICO FORWARD CHAINING ===")
print(formatar_tabela(trilha_fc))

meta_auditoria = "trip_esteira_m101"
sucesso_bc, trilha_bc = motor.backward_chaining(meta_auditoria, fatos_entrada)

print(f"\n=== AUDITORIA BACKWARD CHAINING (Meta: '{meta_auditoria}') ===")
for linha in trilha_bc:
    print(linha)

assert "trip_esteira_m101" in fatos_finais
assert sucesso_bc is True
print("\n[OK] Motores Forward e Backward Chaining validados com sucesso!")